# 🚀 Qpic YOLOv8 Training Notebook
Welcome to the official training notebook for fine-tuning Qpic's Local ML detector on Google Colab using a free GPU.

### 📌 Workflow:
1. **Export training data** from your local Qpic app to a ZIP file.
2. **Upload** the ZIP file to Google Colab (or Google Drive).
3. **Run this notebook** to train the YOLOv8 model for 25-50 epochs on GPU.
4. **Download the trained model** (`best.pt`) and install it back in Qpic.

## 🛠️ Step 1: Install Dependencies
We need to install the `ultralytics` package for training YOLOv8 and `pyyaml` for handling dataset configurations.

In [ ]:
!pip install -q ultralytics>=8.3.0 pyyaml pillow

## 📂 Step 2: Upload Dataset
You have two options to bring your dataset into Colab:

### Option A: Upload directly using the file browser (Left sidebar)
1. Zip your YOLO dataset folder (`temp/custom_yolo` or `temp/hilex_yolo`) into `dataset.zip`.
2. Click the folder icon on the left panel of Colab.
3. Drag and drop `dataset.zip` there.
4. Run the cell below to unzip it.

### Option B: Mount Google Drive (Recommended for large datasets)
Run the Drive mount cell below to access your dataset directly from Google Drive.

In [ ]:
# Run this ONLY if you uploaded a ZIP file directly to Colab
import os
if os.path.exists("dataset.zip"):
    !unzip -q dataset.zip -d /content/dataset
    print("✅ Dataset unzipped successfully!")
else:
    print("❌ dataset.zip not found in /content. Skip if using Google Drive.")

In [ ]:
# Run this ONLY if you want to connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

## 🔍 Step 3: Verify data.yaml
Make sure your `data.yaml` has the correct absolute paths. Let's configure it dynamically for Colab.

In [ ]:
import yaml
from pathlib import Path

# Set the path to your dataset folder in Colab
# Change this if your dataset path is different (e.g., from Drive: "/content/drive/MyDrive/custom_yolo")
dataset_dir = Path("/content/dataset")

yaml_path = dataset_dir / "data.yaml"
if yaml_path.exists():
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    
    # Update paths to be absolute within Colab environment
    data["path"] = str(dataset_dir.resolve())
    data["train"] = "images/train"
    data["val"] = "images/val"
    data["test"] = "images/test"
    
    with open(yaml_path, "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)
    print("✅ data.yaml paths updated for Colab:")
    print(yaml.safe_dump(data))
else:
    print(f"❌ data.yaml not found at {yaml_path}. Make sure dataset is uploaded/unzipped!")

## 🔥 Step 4: Start YOLOv8 GPU Training
Make sure your Colab Runtime is running on GPU! 
To check/change it: **Runtime** -> **Change runtime type** -> select **T4 GPU**.

We will run for **50 epochs** to achieve high accuracy. Feel free to adjust epochs based on your time constraints.

In [ ]:
import torch
from ultralytics import YOLO

# Verify GPU accessibility
device = "0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load pre-trained weights
model = YOLO("yolov8s.pt")

# Start training
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project="/content/runs/detect",
    name="qpic_custom",
    workers=2,
)

## 📊 Step 5: Check Accuracy Metrics
Once training completes, view the charts and mAP (mean Average Precision) scores below.

In [ ]:
from IPython.display import Image, display
import os

run_dir = "/content/runs/detect/qpic_custom"

if os.path.exists(run_dir):
    print("📈 Training curves:")
    display(Image(filename=f"{run_dir}/results.png"))
    
    print("🎯 Confusion Matrix:")
    display(Image(filename=f"{run_dir}/confusion_matrix.png"))
    
    # Print final validation metrics
    metrics = results.results_dict
    print("\n--- 🎯 Model Performance Summary ---")
    print(f"mAP50 (Detection Accuracy): {metrics.get('metrics/mAP50(B)', 0.0):.4f}")
    print(f"Precision: {metrics.get('metrics/precision(B)', 0.0):.4f}")
    print(f"Recall: {metrics.get('metrics/recall(B)', 0.0):.4f}")
else:
    print("❌ Run folder not found. Check if training completed successfully.")

## 💾 Step 6: Save and Download Trained Weights
Let's copy the best model to Google Drive or create a link to download it directly.

In [ ]:
from google.colab import files
import shutil

best_weights = "/content/runs/detect/qpic_custom/weights/best.pt"

if os.path.exists(best_weights):
    # Option A: Download directly to your browser
    print("⬇️ Initiating direct browser download...")
    files.download(best_weights)
    
    # Option B: Save to Google Drive if mounted
    drive_dest = "/content/drive/MyDrive/qpic_best_model.pt"
    if os.path.exists("/content/drive"):
        shutil.copy(best_weights, drive_dest)
        print(f"💾 Also saved a backup copy to Google Drive: {drive_dest}")
else:
    print("❌ best.pt weights not found. Make sure training finished successfully.")